In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
#ω_a = 28.25396662345805
ω_a = 25.338776456203686

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
#ω_a = 3.85*(rand(1)[1]/5+0.9)*2*pi
ω_b_0 = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.55
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=6.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
2E_J*sin(0.8)*φ_b^2

In [ ]:
t_max= 30/κ_b
t_min = 0.0

re_a_0, im_a_0 = 2, 2
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

tspan = (t_min, t_max) # Time span for the simulation

#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
dt_save=2pi/ω_a
t_save = t_max-30/κ_b:dt_save:t_max 
u_0_tot = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = default_solver(u_0_tot, tspan, p, dt; equation=default_equation!, t_save = t_save)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[1:end], q_CL_a[1:end], t_list_CL[1:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

# Large sweep

In [ ]:
ϵ_0_list = [0.4]#range(start = 0.11, stop = 1, length = 5);


t_max = 600pi / (ω_a)#1 / (50 * ω_a)#130/κ_b#
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30*sqrt(2)*ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
(2pi/ω_a)/(1/50ω_a)

In [ ]:
using JSON
satellite_position_dict = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_12_02_2026_Hill.json")
satellite_buffer_position_dict = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_buffer_12_02_2026_Hill.json")
Re_a_list_satellite = []
Im_a_list_satellite = []
for entry in satellite_position_dict
    push!(Re_a_list_satellite,entry["real"]/φ_a)
    push!(Im_a_list_satellite,entry["imag"]/φ_a)
end
Re_b_list_satellite = []
Im_b_list_satellite = []
for entry in satellite_buffer_position_dict
    push!(Re_b_list_satellite,entry["real"]/φ_b)
    push!(Im_b_list_satellite,entry["imag"]/φ_b)
end

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = (re_i + 1im*im_j)

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)
            b_rot = b.*exp.(1im*ω_b_0*t_list)

            Final_mode_a[(re_i,im_j)] = a_rot
            Final_mode_b[(re_i,im_j)] = b_rot
        end
    end
    Final_all_a[ϵ_0] = Final_mode_a
    Final_all_b[ϵ_0] = Final_mode_b
end

In [ ]:
Im_a_list=Im_a_list_satellite
Re_a_list=Re_a_list_satellite
Im_b_list=Im_b_list_satellite
Re_b_list=Re_b_list_satellite
Final_all_a_bis = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b_bis = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=4*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 75:76
        re_i = Re_a_list[i]
        j=i
    
        im_j = Im_a_list[j]
        re_b_i = Re_b_list[i]
        im_b_i = Im_b_list[i]
        a_0_j = (re_i + 1im*im_j)*exp(-1im*0.05)
        b_0_j = (re_b_i + 1im*im_b_i)*exp(-1im*0.1)

        system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0_j)
        system_prepared = prepare(system)

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        a_rot = a.*exp.(1im*ω_a*t_list)
        b_rot = b.*exp.(1im*ω_b_0*t_list)

        Final_mode_a[(re_i,im_j)] = a_rot
        Final_mode_b[(re_i,im_j)] = b_rot
    end
    Final_all_a_bis[ϵ_0] = Final_mode_a
    Final_all_b_bis[ϵ_0] = Final_mode_b
end

In [ ]:
min_re = 0
max_re = 0
min_im = 0
max_im = 0
for ϵ_0 in ϵ_0_list
    Final_mode = Final_all[ϵ_0]
    for (re_i, im_j) in keys(Final_mode)
        array = Final_mode[(re_i, im_j)]
        if minimum(real.(array))< min_re 
            min_re = minimum(real.(array))
        end
        if maximum(real.(array))> max_re 
            max_re = maximum(real.(array))
        end
        if minimum(imag.(array))< min_im
            min_im = minimum(imag.(array))
        end
        if maximum(imag.(array))> max_im 
            max_im = maximum(imag.(array))
        end
    end
end
min_re, max_re, min_im, max_im

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:black)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end
    Final_mode = Final_all_a_bis[ϵ_0]

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:blue)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          #xlims=(min_re, max_re),
          #ylims=(min_im, max_im),
          top_margin=5mm)
    display(p0)
end


In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_b[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:black)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end
    Final_mode = Final_all_b_bis[ϵ_0]

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:blue)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a_bis[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:black)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_b_bis[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=.5, label="", color=:black)
        #plot!(p0,real.(Final_mode[(re_i, im_j)]),imag.(Final_mode[(re_i, im_j)]),arrow=true,color=:black,linewidth=.1,label="")
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
ϵ_d_list = 4:0.2:13.5

t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 3*dt_save
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(sqrt(2)*ω_a)*3
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_d_ratio in ϵ_d_list
    g2 = besselj(1,0.6)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=ϵ_d_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, 0.6, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)

            Final_mode[(re_i,im_j)] = a_rot
        end
    end
    Final_all[ϵ_d_ratio] = Final_mode
end

In [ ]:
initial_value = []
for ϵ_d_ratio in ϵ_d_list
    Final_mode = Final_all[ϵ_d_ratio]
    already = 0
    for (re_i, im_j) in keys(Final_mode)
        if already == 0 
            if real.(Final_mode[(re_i, im_j)])[end]>2.5
                push!(initial_value, (re_i, im_j))
                already = 1
            end
        end
    end
end
initial_value

In [ ]:
length(ϵ_d_list)

In [ ]:
for ϵ_d_ratio in ϵ_d_list
    g2 = besselj(1,0.6)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_d_ratio]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(min_re, max_re),
          ylims=(min_im, max_im),
          top_margin=5mm)

    display(p0)
end


# Fine sweep

In [ ]:
ϵ_0_list = range(start = 0.6044444444444445, stop = 0.802, length = 5);


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(sqrt(2)*ω_a)*3
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            a_rot = a.*exp.(1im*ω_a*t_list)

            Final_mode[(re_i,im_j)] = a_rot
        end
    end
    Final_all[ϵ_0] = Final_mode
end

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-15,15),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end


In [ ]:
using JSON
data_lyapunov = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/ouput/data_lyapunov_no_pump_frequency_29_11_2025.json")
n_rows = 20
n_cols = 20

Im_a_list_lyap=range(-2,2,n_rows)
Re_a_list_lyap=range(-6,0,n_cols)
Lyap_coef_dict = Dict(parse(Float64,k) => reshape(v, n_rows, n_cols) for (k,v) in data_lyapunov)
Lyap_coef_clean = Dict(
    k => replace(v, nothing => NaN)
    for (k, v) in Lyap_coef_dict
)

Lyap_coef_sign = Dict(k => map(x -> x>0 ? 1 : -1, v) for (k, v) in Lyap_coef_clean);

In [ ]:
for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Lyap_coef = Lyap_coef_sign[g2/2pi*1e3]
    Final_mode = Final_all[ϵ_0]

    # Plot heatmap first
    p0 = heatmap(Re_a_list_lyap, Im_a_list_lyap, Lyap_coef,
                 color=:bluesreds, colorbar=false)

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-15,15),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end

# Poincaré section at $\omega_a$

In [ ]:
ϵ_0_list = range(start = 0.576, stop = 0.612, length = 10);


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_0 in ϵ_0_list
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode_a[(re_i,im_j)] = a
            Final_mode_b[(re_i,im_j)] = b
        end
    end
    Final_all_a[ϵ_0] = Final_mode_a
    Final_all_b[ϵ_0] = Final_mode_b
end

In [ ]:
for ϵ_0 in ϵ_0_list

    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-10,10),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end


In [ ]:
for ϵ_0 in ϵ_0_list

    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_b[ϵ_0]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(b)",
          ylabel=L"\Im(b)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          xlims=(-10,10),
          ylims=(-6,6),
          top_margin=5mm)

    display(p0)
end


In [ ]:
κ_a_list = 10.0 .^range(start = -1.4, stop = -1.2, length = 15)


t_max = 1000/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.72
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for κ_a in [0.0]
    environment = MeanField(κ_a, κ_b)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment);

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode_a[(re_i,im_j)] = a
            Final_mode_b[(re_i,im_j)] = b
        end
    end
    Final_all_a[κ_a] = Final_mode_a
    Final_all_b[κ_a] = Final_mode_b
end

In [ ]:
for κ_a in [0.0]
    Final_mode = Final_all_a[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"\kappa_a="*string(κ_a)*" GHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
for κ_a in kappa_a_list

    Final_mode = Final_all_a[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
for κ_a in kappa_a_list

    Final_mode = Final_all_b[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(b)",
          ylabel=L"\Im(b)",
          title=L"\kappa_a="*string(κ_a)*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
function orbit_cardinality(z::AbstractVector{<:Complex}; eps=1e-2)
    reps = ComplexF64[]
    for zi in z
        if all(abs(zi - r) > eps for r in reps)
            push!(reps, zi)
        end
    end
    return length(reps)
end
function run_notebook(κ_a::Float64)
    environment = MeanField(κ_a_max, κ_b)
    environment_prepared = environment;
    Final_mode_b = Vector{ComplexF32}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            _, b, _ = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            append!(Final_mode_b, b)
        end
    end
    return Final_mode_b
end


In [ ]:
κ_a_min = 0.01
κ_a_max = 0.02


t_max = 250/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)


In [ ]:
gap = 1e-4
res_dict = Dict{Float64, Float64}()
Final_mode_b = Vector{ComplexF64}()
while abs(κ_a_max - κ_a_min) > gap
    κ_a_new = (κ_a_min + κ_a_max) / 2
    Final_mode_b = run_notebook(κ_a_new)
    mask = (-1 .<= real.(Final_mode_b) .<= -.8) .&
       (-3 .<= imag.(Final_mode_b) .<= -2)
    res = orbit_cardinality(Final_mode_b[mask])
    res_dict[κ_a_new] = res
    if  res == 2
        κ_a_max = κ_a_new
    elseif res > 6
        κ_a_min = κ_a_new
    else 
        break
    end
end
print(res_dict)

In [ ]:
mask = (0 .<= real.(Final_mode_b) .<= 1) .&
    (0 .<= imag.(Final_mode_b) .<= 1.5)
scatter(real.(Final_mode_b), imag.(Final_mode_b))

In [ ]:
κ_a_list[end-7:end-6]

In [ ]:
kappa_a_list = κ_a_list = 0.04648858351288666 , stop = 0.046488583512886664, length = 11)


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment);

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode_a[(re_i,im_j)] = a
            Final_mode_b[(re_i,im_j)] = b
        end
    end
    Final_all_a[κ_a] = Final_mode_a
    Final_all_b[κ_a] = Final_mode_b
end

In [ ]:
for κ_a in kappa_a_list

    Final_mode = Final_all_a[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_mod_1 = 1:4:length(z)
        idx_mod_2  = 2:4:length(z)
        idx_mod_3 = 3:4:length(z)
        idx_mod_4  = 4:4:length(z)
        scatter!(
            p0,
            real.(z[idx_mod_1]),
            imag.(z[idx_mod_1]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_mod_2]),
            imag.(z[idx_mod_2]),
            ms = 1,
            color = :red,
            label = "",
        )
                scatter!(
            p0,
            real.(z[idx_mod_3]),
            imag.(z[idx_mod_3]),
            ms = 1,
            color = :green,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_mod_4]),
            imag.(z[idx_mod_4]),
            ms = 1,
            color = :orange,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"\kappa_a="*string(κ_a)*" GHz",
          xlims=(2,4), ylims=(-1,3),
          top_margin=5mm)

    display(p0)
end


In [ ]:
for κ_a in kappa_a_list

    Final_mode = Final_all_b[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_mod_1 = 1:4:length(z)
        idx_mod_2  = 2:4:length(z)
        idx_mod_3 = 3:4:length(z)
        idx_mod_4  = 4:4:length(z)
        scatter!(
            p0,
            real.(z[idx_mod_1]),
            imag.(z[idx_mod_1]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_mod_2]),
            imag.(z[idx_mod_2]),
            ms = 1,
            color = :red,
            label = "",
        )
                scatter!(
            p0,
            real.(z[idx_mod_3]),
            imag.(z[idx_mod_3]),
            ms = 1,
            color = :green,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_mod_4]),
            imag.(z[idx_mod_4]),
            ms = 1,
            color = :orange,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(b)",
          ylabel=L"\Im(b)",
          title=L"\kappa_a="*string(κ_a)*" GHz",
          top_margin=5mm)

    display(p0)
end


# STD plots

In [ ]:
values_a = Vector{ComplexF64}()
dict = Final_all[0.802]
for val in values(dict)
    append!(values_a, val)
end

In [ ]:
30/κ_b/dt_save

In [ ]:
x = real.(values_a)
y = imag.(values_a)

histogram2d(
    x, y;
    bins = 100,
    cmap = :viridis,
    xlabel = L"\Re(a)",
    ylabel = L"\Im(a)",
    title = "Density",
    aspect_ratio = 1,
    colorbar = true,
)

In [ ]:
using Statistics
dict = Final_all[0.802]
std_vec = Vector{Float64}()
for val in values(dict)
    append!(std_vec, std(val))
end
histogram(
    std_vec;
    bins = 500,
    cmap = :viridis,
    xlabel = "Std",
    colorbar = true,
)

In [ ]:
length(values_a)

In [ ]:
using Statistics

group_size = 1000
dict = Final_all[0.802]
trajectories = collect(values(dict))
ntraj = length(trajectories)

# number of chunks (assume same length; otherwise take minimum)
nchunks = minimum(length(traj) ÷ group_size for traj in trajectories)

std_per_chunk = zeros(Float64, nchunks, ntraj)

for (j, traj) in enumerate(trajectories)
    for k in 1:nchunks
        i0 = (k-1)*group_size + 1
        i1 = k*group_size
        std_per_chunk[k, j] = std(@view traj[i0:i1])
    end
end


In [ ]:
using StatsBase

nbins = 100
std_min = minimum(std_per_chunk)
std_max = maximum(std_per_chunk)

std_bins = range(std_min, std_max; length=nbins+1)
std_centers = (std_bins[1:end-1] .+ std_bins[2:end]) ./ 2

In [ ]:
hist_matrix = zeros(Float64, nbins, nchunks)

for k in 1:nchunks
    h = fit(Histogram, std_per_chunk[k, :], std_bins)
    hist_matrix[:, k] .= h.weights
end


In [ ]:
chunk_indices = (0:nchunks-1) .* group_size .+ 1   # index of first point in chunk
chunk_time_converted = chunk_indices*dt_save*κ_b

heatmap(
    chunk_time_converted,
    std_centers,
    hist_matrix;
    cmap = :viridis,
    xlabel = L"\kappa_bt",
    ylabel = "Std",
    colorbar = true,
    ylims = (-0.0, 3.8)
)